In [1]:
import json
import pickle
import os

print("✅ Ready to build catalog!")

✅ Ready to build catalog!


In [2]:
DATA_DIR = '../data/ddxplus'

with open(os.path.join(DATA_DIR, 'release_evidences.json'), encoding='utf-8') as f:
    evidences = json.load(f)

with open('../models/symptom_columns.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

print(f"✅ Loaded {len(feature_cols)} features to describe")
print(f"   Evidence definitions available: {len(evidences)}")

✅ Loaded 270 features to describe
   Evidence definitions available: 223


In [3]:
def describe_feature(feat):
    
    if feat == 'AGE':
        return {"code": "AGE", "type": "numeric",
                "label": "Age (years)", "group": "Demographics"}
    if feat == 'SEX_M':
        return {"code": "SEX_M", "type": "binary",
                "label": "Sex: Male", "group": "Demographics"}
    if feat == 'SEX_F':
        return {"code": "SEX_F", "type": "binary",
                "label": "Sex: Female", "group": "Demographics"}

    
    if '_@_' in feat:
        ev_code, val_code = feat.split('_@_', 1)
        ev        = evidences.get(ev_code, {})
        question  = ev.get('question_en', ev_code)
        vmeaning  = ev.get('value_meaning', {})
        val_label = vmeaning.get(val_code, {}).get('en', val_code)
        is_ant    = ev.get('is_antecedent', False)
        return {"code": feat, "type": "categorical", "evidence": ev_code,
                "question": question, "value": val_label,
                "label": f"{question}  →  {val_label}",
                "group": "Antecedents" if is_ant else "Symptoms"}

    
    ev       = evidences.get(feat, {})
    question = ev.get('question_en', feat)
    is_ant   = ev.get('is_antecedent', False)
    return {"code": feat, "type": "binary", "question": question,
            "label": question,
            "group": "Antecedents" if is_ant else "Symptoms"}

feature_metadata = {feat: describe_feature(feat) for feat in feature_cols}

print("=== Sample of readable labels ===\n")
for feat in feature_cols[:12]:
    print(f"  {feat:<22} → {feature_metadata[feat]['label']}")

=== Sample of readable labels ===

  E_0                    → Have you recently had a viral infection?
  E_1                    → Are you currently being treated or have you recently been treated with an oral antibiotic for an ear infection?
  E_10                   → Are you currently taking or have you recently taken anti-inflammatory drugs (NSAIDs)?
  E_101                  → Have you been hospitalized for an asthma attack in the past year?
  E_103                  → Have you lost your sense of smell?
  E_104                  → Do you have high blood pressure or do you take medications to treat high blood pressure?
  E_105                  → Have you ever had a heart attack or do you have angina (chest pain)?
  E_11                   → Have you breastfed one of your children for more than 9 months?
  E_111                  → Do you feel like you are dying or were you afraid that you were about do die?
  E_112                  → Do you wheeze while inhaling or is your breathing noisy

In [4]:
with open('../models/feature_metadata.pkl', 'wb') as f:
    pickle.dump(feature_metadata, f)


with open('../models/feature_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(feature_metadata, f, ensure_ascii=False, indent=2)


from collections import Counter
groups = Counter(m['group'] for m in feature_metadata.values())

print("✅ Catalog saved!")
print(f"   models/feature_metadata.pkl")
print(f"   models/feature_metadata.json")
print(f"\n   Breakdown: {dict(groups)}")

✅ Catalog saved!
   models/feature_metadata.pkl
   models/feature_metadata.json

   Breakdown: {'Antecedents': 72, 'Symptoms': 195, 'Demographics': 3}
